# Práctica 3: Resolución de un problema usando aprendizaje automático
# Proyecto: Predicción de Emisiones de CO2 en Vehículos

**Autores:** Daniel Martín del Castillo, PONED EL NOMBRE!!! 
**Objetivo:** El objetivo de este notebook es desarrollar un modelo de clasificación capaz de predecir las emisiones de CO2 (g/km) de un vehículo basándose en sus características técnicas y su consumo de combustible.

In [ ]:
# Importamos las librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargamos el dataset
df = pd.read_csv('co2.csv')

# Visualizamos las primeras filas para confirmar que todo está bien
df.head()

# Análisis Exploratorio de Datos (EDA)
En esta sección analizamos la estructura del dataset para identificar posibles valores nulos, tipos de variables y la relación visual entre las características del motor y las emisiones.

In [ ]:
# Información técnica y nulos
print("--- Información del Dataset ---")
df.info()

# Gráfico de relación entre el tamaño del motor vs emisiones
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df, x='Engine Size(L)', y='CO2 Emissions(g/km)', hue='Fuel Type', alpha=0.6)
plt.title('Relación: Tamaño del Motor vs Emisiones de CO2')
plt.show()

# Pre-procesamiento y Transformación
En esta fase preparamos los datos siguiendo la estrategia definida en nuestros experimentos previos:
1. **Categorización**: Convertimos las emisiones de CO2 (numéricas) en categorías de impacto ambiental.
2. **Limpieza**: Eliminamos columnas que no aportan valor predictivo o son redundantes.
3. **División**: Separamos los datos en entrenamiento (80%) y test (20%) para evaluar el rendimiento real.

# Limpieza de Datos y Selección de Variables
Tras analizar el dataset original, hemos procedido a eliminar ciertas columnas que podrían perjudicar el rendimiento de nuestros modelos o que resultan redundantes para el objetivo de la predicción:

**Eliminación de la columna Model**: Esta variable contiene el nombre específico de cada modelo de coche. Al haber miles de modelos distintos, incluirla en el modelo obligaría a crear miles de columnas nuevas durante la codificación, lo que haría el entrenamiento excesivamente lento e ineficiente.

**Eliminación de Fuel Consumption Comb (mpg)**: El dataset ya incluye el consumo de combustible en litros por cada 100 km (L/100 km). Mantener ambas columnas es inecesario, ya que ambas representan la misma información en unidades distintas. Hemos optado por mantener la métrica europea (L/100 km) por ser más estándar en nuestro contexto.  

**Eliminación de CO2 Emissions(g/km)**: Al haber transformado el problema en uno de clasificación, eliminamos la variable numérica original para evitar que el modelo tenga acceso directo al valor que queremos predecir en su versión categórica.  

Con esta limpieza, nos aseguramos de que el modelo trabaje únicamente con variables que aportan valor real y facilitan el aprendizaje de patrones generales sobre las marcas y tipos de vehículos.

In [ ]:
# Categorizacion del CO2
bins   = [0, 120, 150, 180, 210, 250, 300, float('inf')]
labels = ['<120', '120-150', '150-180', '180-210', '210-250', '250-300', '>300']

df['CO2_categoria'] = pd.cut(df['CO2 Emissions(g/km)'], bins=bins, labels=labels).astype(str)

# Limpieza de columnas
df_modelo = df.drop(columns=['CO2 Emissions(g/km)', 'Model', 'Fuel Consumption Comb (mpg)'])

# Division del dataset
from sklearn.model_selection import train_test_split

X = df_modelo.drop(columns=['CO2_categoria'])
y = df_modelo['CO2_categoria']

x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=1224)

print(f"Dataset dividido: {x_train.shape[0]} muestras de entreno y {x_test.shape[0]} de test.")
x_train.head()

# Codificación de Variables Categóricas
Para que los algoritmos de aprendizaje automático puedan procesar la información de las variables de texto (como la marca o el tipo de combustible en nuestro caso), utilizamos **OneHotEncoder**. Esta técnica crea columnas binarias para cada categoría, permitiendo que el modelo interprete estas características sin asumir un orden jerárquico.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Definimos las columnas categoricas a transformar
columnas_cat = ["Make", "Vehicle Class", "Transmission", "Fuel Type"]

# Inicializamos el encoder eliminando la primera columna de cada categoría para evitar redundancia
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Entrenamos el encoder con los datos de entrenamiento
encoder.fit(x_train[columnas_cat])

# Transformamos tanto el conjunto de entrenamiento como el de prueba
train_enc = encoder.transform(x_train[columnas_cat])
test_enc = encoder.transform(x_test[columnas_cat])

# Obtenemos los nombres de las nuevas columnas generadas
cols_nombres = encoder.get_feature_names_out(columnas_cat)

# Creamos los DataFrames finales uniendo las variables numéricas con las nuevas columnas codificadas
x_train_final = x_train.drop(columns=columnas_cat).reset_index(drop=True).join(pd.DataFrame(train_enc, columns=cols_nombres))
x_test_final = x_test.drop(columns=columnas_cat).reset_index(drop=True).join(pd.DataFrame(test_enc, columns=cols_nombres))

print(f"El dataset ahora tiene {x_train_final.shape[1]} columnas.")
x_train_final.head()

# Experimentación y Comparativa de Modelos
En esta fase procedemos a la experimentación con diferentes tecnologías de aprendizaje automático. El objetivo es evaluar qué algoritmo se adapta mejor a la naturaleza de nuestro dataset de emisiones de CO2 una vez categorizado. 
Para garantizar una comparativa correcta, seguiremos los siguientes pasos:

**-Selección de Tecnologías**: Hemos seleccionado los siguientes 4 algoritmos:
    **Decision Tree (Árbol de Decisión)**: Un modelo basado en reglas lógicas que destaca por su interpretabilidad.  
    **Random Forest**: Un modelo de "ensamble" que combina múltiples árboles para mejorar la precisión y reducir el sobreajuste.  
    **K-Nearest Neighbors (KNN)**: Un algoritmo basado en la "cercanía" o similitud geométrica entre los datos.  
    **Regresión Logística**: Un modelo lineal clásico utilizado como base de referencia para problemas de clasificación multiclase.  
    
**-Métrica de Evaluación**: Utilizaremos el Accuracy (Precisión) como medida principal para cuantificar qué porcentaje de vehículos son clasificados correctamente en su rango de emisiones correspondiente.  

**-Validación**: Todos los modelos se entrenarán con el 80% de los datos y se evaluarán con el 20% restante, que el modelo nunca ha visto, asegurando que los resultados sean representativos de su desempeño real.  

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Definimos los modelos con la configuración que mejor resultados dio al equipo
modelos = {
    "Decision Tree": DecisionTreeClassifier(max_depth=15, random_state=1224),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, random_state=1224),
    "KNN": KNeighborsClassifier(n_neighbors=10),
    "Regresión Logística": LogisticRegression(max_iter=1000)
}

resultados = {}

print("--- Iniciando entrenamiento de los modelos ---")

for nombre, modelo in modelos.items():
    # El modelo estudia los datos de entrenamiento (91 columnas)
    modelo.fit(x_train_final, y_train)
    
    # El modelo intenta adivinar las categorías del conjunto de test
    y_pred = modelo.predict(x_test_final)
    
    # Comparamos sus aciertos con la realidad
    acc = accuracy_score(y_test, y_pred)
    resultados[nombre] = acc
    print(f"{nombre}: Accuracy = {acc:.4f}")

# Generamos el Ranking final
ranking = sorted(resultados.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*30)
print("RANKING FINAL DE TECNOLOGÍAS")
print("="*30)
for i, (nombre, acc) in enumerate(ranking, 1):
    print(f"{i}. {nombre:<20} | Precisión: {acc:.4f}")

# Análisis Técnico 
Durante la fase de experimentación, el modelo de Regresión Logística mostró un aviso de tipo **ConvergenceWarning** tras alcanzar el límite de 1000 iteraciones. Esto nos informa de:  
    **-No Linealidad del Problema**: La Regresión Logística intenta encontrar una separación lineal entre las categorías de CO2. El hecho de que el algoritmo no encuentre la solución óptima nos hace ver que la relación entre las especificaciones técnicas del vehículo y sus emisiones es compleja y no lineal.  
    **-Superioridad de los Árboles**: Esto explica por qué el Decision Tree y el Random Forest han obtenido resultados significativamente superiores (aprox. 96% frente al 89%). Los árboles de decisión no intentan trazar una línea, sino que realizan divisiones del espacio de datos, lo que les permite capturar mejor esas relaciones.  
    **-Elección**: Este resultado nos sirve para descartar los modelos lineales para este dataset específico.  La experimentación con múltiples tecnologías nos ha permitido identificar que los modelos basados en árboles son los más adecuados para la naturaleza de los datos de emisiones vehiculares.  